In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch
import networkx as nx
import matplotlib.pyplot as plt


# =========================
# 1) 读取 dump 并构建索引
# =========================
def load_manifest(dump_dir: str):
    with open(os.path.join(dump_dir, "manifest.json"), "r", encoding="utf-8") as f:
        return json.load(f)

def _resolve_layer(mani, layer):
    n_layers = mani["n_layers"]
    return n_layers + layer if layer < 0 else layer

def build_edge_indices_from_dump(
    dump_dir: str,
    layer: int = -1,
    use_head: bool = False,
    head_agg: str = "mean",   # "mean" or "sum"
):
    """
    读取某一层的所有 attention 文件，构建：
      in_index[(srctype,etype,dsttype)][dst_id] -> list of (src_id, weight)
      out_index[(srctype,etype,dsttype)][src_id] -> list of (dst_id, weight)
    以及 node_types 供上色：("ntype", nid) -> ntype
    """
    mani = load_manifest(dump_dir)
    layer = _resolve_layer(mani, layer)

    in_index = {}   # key: c_etype -> dict(dst -> list[(src,w)])
    out_index = {}  # key: c_etype -> dict(src -> list[(dst,w)])

    for item in mani["files"]:
        if item["layer"] != layer:
            continue
        srctype, etype, dsttype = item["canonical_etype"]
        c_etype = (srctype, etype, dsttype)

        obj = torch.load(item["path"], map_location="cpu")
        src = obj["src"].numpy().astype(np.int64)
        dst = obj["dst"].numpy().astype(np.int64)

        if use_head:
            attn_head = obj["attn_head"]  # (E,H)
            if head_agg == "mean":
                w = attn_head.mean(dim=1).numpy()
            elif head_agg == "sum":
                w = attn_head.sum(dim=1).numpy()
            else:
                raise ValueError("head_agg must be mean or sum")
        else:
            w = obj["attn_mean"].numpy()

        # init dicts
        if c_etype not in in_index:
            in_index[c_etype] = {}
            out_index[c_etype] = {}

        # build indices
        for s, d, ww in zip(src, dst, w):
            in_index[c_etype].setdefault(int(d), []).append((int(s), float(ww)))
            out_index[c_etype].setdefault(int(s), []).append((int(d), float(ww)))

    # sort each adjacency list by weight desc
    for c_etype in in_index:
        for d in in_index[c_etype]:
            in_index[c_etype][d].sort(key=lambda x: x[1], reverse=True)
        for s in out_index[c_etype]:
            out_index[c_etype][s].sort(key=lambda x: x[1], reverse=True)

    return mani, layer, in_index, out_index


# =========================
# 2) 从 patient 扩展子图（包含组学间关系）
# =========================
def collect_subgraph_around_patient(
    mani,
    in_index,
    out_index,
    patient_nid: int,
    hops: int = 2,
    topk_in: int = 30,
    topk_out: int = 30,
    weight_thresh: float = 0.0,
):
    """
    BFS 扩展：
      - 对当前 frontier 的每个节点，收集其 topk 入边和 topk 出边（按权重）
      - 这样既能包含 “组学 -> patient”，也能包含 “组学之间调控/连接”
    返回：
      nodes: set of (ntype, nid)
      edges: list of dicts {u=(ntype,id), v=(ntype,id), weight, etype=(srctype,etype,dsttype)}
    """
    # helper: list canonical etypes
    canonical_etypes = [tuple(x) for x in mani["canonical_etypes"]]

    start = ("patient", int(patient_nid))
    nodes = {start}
    frontier = {start}
    edges = []

    for _ in range(hops):
        new_frontier = set()

        for (ntype, nid) in list(frontier):
            # 1) incoming edges (src -> this node)
            for c_etype in canonical_etypes:
                srctype, etype, dsttype = c_etype
                if dsttype != ntype:
                    continue
                if c_etype not in in_index:
                    continue
                nbrs = in_index[c_etype].get(nid, [])
                for s_id, w in nbrs[:topk_in]:
                    if w < weight_thresh:
                        continue
                    u = (srctype, int(s_id))
                    v = (dsttype, int(nid))
                    nodes.add(u); nodes.add(v)
                    edges.append({"u": u, "v": v, "weight": float(w), "etype": c_etype})
                    new_frontier.add(u)

            # 2) outgoing edges (this node -> dst)
            for c_etype in canonical_etypes:
                srctype, etype, dsttype = c_etype
                if srctype != ntype:
                    continue
                if c_etype not in out_index:
                    continue
                nbrs = out_index[c_etype].get(nid, [])
                for d_id, w in nbrs[:topk_out]:
                    if w < weight_thresh:
                        continue
                    u = (srctype, int(nid))
                    v = (dsttype, int(d_id))
                    nodes.add(u); nodes.add(v)
                    edges.append({"u": u, "v": v, "weight": float(w), "etype": c_etype})
                    new_frontier.add(v)

        frontier = new_frontier

    # 去重（同一条边可能重复出现）
    uniq = {}
    for e in edges:
        key = (e["u"][0], e["u"][1], e["etype"][1], e["v"][0], e["v"][1])
        # 保留最大权重
        if key not in uniq or e["weight"] > uniq[key]["weight"]:
            uniq[key] = e
    edges = list(uniq.values())

    return nodes, edges


# =========================
# 3) 可视化（节点颜色=组学类型，边深浅/粗细=权重）
# =========================
def visualize_subgraph(
    nodes,
    edges,
    out_png: str,
    out_csv: str,
    title: str = None,
    max_nodes: int = 250,
    seed: int = 0,
):
    """
    - 节点颜色按 ntype
    - 边 alpha/width 按 weight
    - 自动生成 legend
    """
    # 如果太大，按“与 patient 的边权”做一次裁剪：保留最高权重相关的节点
    # 简化策略：按节点 incident edge 的最大权重排序截断
    if len(nodes) > max_nodes:
        score = {n: 0.0 for n in nodes}
        for e in edges:
            score[e["u"]] = max(score.get(e["u"], 0.0), e["weight"])
            score[e["v"]] = max(score.get(e["v"], 0.0), e["weight"])
        keep = set(sorted(score.keys(), key=lambda x: score[x], reverse=True)[:max_nodes])
        nodes = keep
        edges = [e for e in edges if e["u"] in keep and e["v"] in keep]

    # build nx graph
    Gnx = nx.DiGraph()
    for (ntype, nid) in nodes:
        Gnx.add_node(f"{ntype}:{nid}", ntype=ntype, nid=nid)

    for e in edges:
        u = f"{e['u'][0]}:{e['u'][1]}"
        v = f"{e['v'][0]}:{e['v'][1]}"
        Gnx.add_edge(u, v, weight=e["weight"], etype="|".join(e["etype"]))

    # save edges csv
    df = pd.DataFrame([{
        "u": f"{e['u'][0]}:{e['u'][1]}",
        "v": f"{e['v'][0]}:{e['v'][1]}",
        "srctype": e["etype"][0],
        "etype": e["etype"][1],
        "dsttype": e["etype"][2],
        "weight": e["weight"],
    } for e in edges]).sort_values("weight", ascending=False)
    df.to_csv(out_csv, index=False)

    # color map by ntype (你也可以按自己 ntype 改这些颜色)
    # 如果出现新类型，会自动分配为灰色
    palette = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
        "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
        "#bcbd22", "#17becf"
    ]
    ntypes = sorted({data["ntype"] for _, data in Gnx.nodes(data=True)})
    color_map = {}
    for i, nt in enumerate(ntypes):
        color_map[nt] = palette[i % len(palette)]
    # patient 节点更显眼
    if "patient" in color_map:
        color_map["patient"] = "#000000"

    node_colors = [color_map[Gnx.nodes[n]["ntype"]] for n in Gnx.nodes()]
    node_sizes = [1400 if Gnx.nodes[n]["ntype"] == "patient" else 260 for n in Gnx.nodes()]

    # layout
    pos = nx.spring_layout(Gnx, seed=seed, k=None)

    # edge style by weight
    ws = np.array([Gnx.edges[e]["weight"] for e in Gnx.edges()], dtype=float)
    if len(ws) == 0:
        print("[WARN] no edges to plot.")
        return

    w_min, w_max = float(ws.min()), float(ws.max())
    w_norm = (ws - w_min) / (w_max - w_min + 1e-12)

    edge_widths = 0.4 + 6.0 * w_norm
    edge_alphas = 0.1 + 0.9 * w_norm  # 深浅
    edge_colors = [(0, 0, 0, float(a)) for a in edge_alphas]  # 黑色+透明度

    plt.figure(figsize=(14, 10))
    nx.draw_networkx_nodes(Gnx, pos, node_color=node_colors, node_size=node_sizes)
    nx.draw_networkx_edges(Gnx, pos, width=edge_widths, edge_color=edge_colors, arrows=True, arrowsize=10)

    # labels：只标 patient + top-weight 节点
    # top 节点按 incident 最大权重
    incident_max = {n: 0.0 for n in Gnx.nodes()}
    for u, v, data in Gnx.edges(data=True):
        w = float(data["weight"])
        incident_max[u] = max(incident_max[u], w)
        incident_max[v] = max(incident_max[v], w)

    # 标注最多 25 个最重要节点 + patient
    top_nodes = sorted(incident_max.keys(), key=lambda x: incident_max[x], reverse=True)[:25]
    labels = {}
    for n in top_nodes:
        labels[n] = n  # 你如果有 name 映射，可以在这里替换成真实名字
    for n in Gnx.nodes():
        if Gnx.nodes[n]["ntype"] == "patient":
            labels[n] = n

    nx.draw_networkx_labels(Gnx, pos, labels=labels, font_size=8)

    # legend
    handles = []
    import matplotlib.patches as mpatches
    for nt in ntypes:
        handles.append(mpatches.Patch(color=color_map[nt], label=nt))
    plt.legend(handles=handles, loc="best", fontsize=9, frameon=True)

    if title:
        plt.title(title)

    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300)
    plt.close()

    print("[SAVED]", out_png)
    print("[SAVED]", out_csv)


# =========================
# 4) 一键函数：给定 patient 画图
# =========================
def visualize_patient_multimodal_network(
    dump_dir: str,
    patient_nid: int,
    layer: int = -1,
    hops: int = 2,
    topk_in: int = 30,
    topk_out: int = 30,
    weight_thresh: float = 0.0,
    max_nodes: int = 250,
    seed: int = 0,
):
    mani, real_layer, in_index, out_index = build_edge_indices_from_dump(
        dump_dir=dump_dir, layer=layer, use_head=False
    )

    nodes, edges = collect_subgraph_around_patient(
        mani=mani,
        in_index=in_index,
        out_index=out_index,
        patient_nid=patient_nid,
        hops=hops,
        topk_in=topk_in,
        topk_out=topk_out,
        weight_thresh=weight_thresh,
    )

    out_png = os.path.join(dump_dir, f"patient_{patient_nid}_layer{real_layer}_hops{hops}.png")
    out_csv = os.path.join(dump_dir, f"patient_{patient_nid}_layer{real_layer}_hops{hops}_edges.csv")

    visualize_subgraph(
        nodes=nodes,
        edges=edges,
        out_png=out_png,
        out_csv=out_csv,
        title=f"Patient {patient_nid} multimodal network (layer={real_layer}, hops={hops})",
        max_nodes=max_nodes,
        seed=seed,
    )


# =========================
# 5) 示例用法
# =========================
if __name__ == "__main__":
    dump_dir = "/data5/zhangye/dgl/examples/pytorch/morn/acc_hgt_dataset/attn_dump_final"
    patient_nid = 12  # 换成你想看的 patient 节点 id

    visualize_patient_multimodal_network(
        dump_dir=dump_dir,
        patient_nid=patient_nid,
        layer=-1,        # 最后一层
        hops=2,          # 2-hop：能包含组学->组学->patient 的链路
        topk_in=30,      # 每个节点最多保留 top 30 入边
        topk_out=30,     # 每个节点最多保留 top 30 出边
        weight_thresh=0.0, # 想更稀疏可以设 0.01 / 0.02
        max_nodes=250,   # 图太大就降
        seed=0,
    )


In [19]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ===== 改这里 =====
csv_path = "/data5/zhangye/morn/attn_compare_out_v2/csv/layer_1/PATH_gene_miRNA__targets__gene_mRNA_top20.csv"
topk = 4
fig_w = 3

out_dir = os.path.dirname(csv_path) if csv_path.endswith(".csv") else os.path.dirname(csv_path.rstrip("*"))
os.makedirs(out_dir, exist_ok=True)
files = [csv_path] if csv_path.endswith(".csv") else sorted(glob.glob(csv_path))
print("[FILES]", len(files))

cmap = plt.get_cmap("viridis")  # viridis / plasma 也行

# ✅ hatch 参数（条纹样式可换：'///', '\\\\', 'xx', 'oo', '..', '++'）
HATCH = "///"
BAR_H = 0.80          # ✅ 调大 -> 条形更“挤”，间距更小（0.85~0.95都行）
EDGE_ALPHA = 0.65     # 边线透明度
EDGE_LW = 0.6         # 边线宽度（hatch 依赖 edgecolor 更明显）

for f in files:
    df = pd.read_csv(f)
    gene_col = "gene"
    title = df["path"].iloc[0] if ("path" in df.columns and len(df)) else os.path.basename(f)
    patients = [c for c in df.columns if c not in ["layer", "path", "gene"]]

    long = df[[gene_col] + patients].melt(gene_col, var_name="patient", value_name="score")
    long["score"] = pd.to_numeric(long["score"], errors="coerce")
    long = long.dropna()

    long = (long.sort_values(["patient", "score"], ascending=[True, False])
                .groupby("patient").head(topk).copy())

    ssum = long.groupby("patient")["score"].transform("sum").replace(0, np.nan)
    long["ratio"] = (long["score"] / ssum).fillna(0.0)

    pats = long["patient"].unique().tolist()
    if not pats:
        print("[SKIP]", f)
        continue

    for pid in pats:
        sub = long[long["patient"] == pid].copy()
        if sub.empty:
            continue

        sub = sub.sort_values("ratio", ascending=True)

        vmin, vmax = float(sub["ratio"].min()), float(sub["ratio"].max())
        if abs(vmax - vmin) < 1e-12:
            vmin, vmax = 0.0, 1.0
        norm = Normalize(vmin=vmin, vmax=vmax)

        xmin = max(0.0, vmin - 0.02)
        xmax = min(1.0, vmax + 0.02)

        y = np.arange(len(sub))
        vals = sub["ratio"].to_numpy()
        facecolors = cmap(norm(vals))

        fig_h = 1 + 0.45 * topk
        fig, ax = plt.subplots(1, 1, figsize=(fig_w, fig_h))

        # ✅ 关键：颜色 + hatch（条纹/形状填充）
        bars = ax.barh(
            y, vals,
            color=facecolors,          # 保留原本颜色
            edgecolor=(0, 0, 0, EDGE_ALPHA),  # hatch 的颜色来自 edgecolor
            linewidth=EDGE_LW,
            hatch=HATCH,
            height=BAR_H               # ✅ 条形更密，间距更小
        )

        ax.set_yticks(y)
        ax.set_yticklabels(sub[gene_col].tolist(), fontsize=11)
        ax.set_title(pid, loc="left", fontsize=12, pad=4)

        ax.set_xlim(xmin, xmax)
        ax.grid(axis="x", alpha=0.22, linewidth=0.7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.set_xlabel(f"Top-{topk} ratio = score / sum(top{topk})", fontsize=12)
        fig.suptitle(title, fontsize=14, y=0.98)

        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, fraction=0.06, pad=0.03)
        cbar.set_label("Ratio (color)", rotation=90)

        fig.tight_layout(rect=[0, 0, 1, 0.93])

        safe_pid = pid.replace("|", "_").replace("/", "_")
        out_png = os.path.join(out_dir, os.path.splitext(os.path.basename(f))[0] + f"_{safe_pid}_top{topk}_ratio_hatch.png")
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print("[SAVE]", out_png)

[FILES] 1
[SAVE] /data5/zhangye/morn/attn_compare_out_v2/csv/layer_1/PATH_gene_miRNA__targets__gene_mRNA_top20_TCGA-OR-A5K5_y=1_top4_ratio_hatch.png
[SAVE] /data5/zhangye/morn/attn_compare_out_v2/csv/layer_1/PATH_gene_miRNA__targets__gene_mRNA_top20_TCGA-OR-A5KO_y=3_top4_ratio_hatch.png
